# Figures

## Load simulation data

In [2]:
include("packages.jl")

filename = "output/sol_2025-11-19_14-39.jld2"

@load filename sol local_tags_list

2-element Vector{Symbol}:
 :sol
 :local_tags_list

## Translation elongation rate profile

### Get ribosome concentrations for a specific sequence from sol

#### Divide sequences in a list of dataframes that have the metabolites for all and the ribosomes for one sequence

In [5]:
df_list = sequence_dataframes(sol, local_tags_list);

#### Calculate smoothed elongation profile

In [12]:
tags = local_tags_list[1]
y = df_list[1][end,2:end] |> collect;

In [27]:
# Calculate q_R_elo 
# necessary parameters 
len_sequence = length((tags.species_and_sequences.Seq))
gene_sizes = length((tags.species_and_sequences.Seq))-1

@unpack µ, k_proc, k_cat_Pol, C_RNP, V_TCmax, K_M_D, k_cat_E, C_RNE, V_degmax, K_M_deg, k_D, n_DNA_t, n_DNA_s = tags.p_seq.TX
@unpack k_TLEmaxj, KM_T3j, KM_EFGGTP = tags.p_seq.EL
@unpack k_TE, KM_RK, KM_GTP, KM_RF3GTP, k_RF3_diss_GDP, k_RF1_diss_GTP, k_GTP_hydr, k_RF1_ass = tags.p_seq.TER
@unpack k_RF3_GDP_ass, k_RF3_GTP_ass, k_RF3_GDP_diss, k_RF3_GTP_diss = tags.p_seq.RF3
@unpack A1, A2, A3, A4, A5, k_cat_K_M_G1, k_cat_K_M_G2, k_RRF, q_RRF, one_q_G2_k_split = tags.p_seq.PTC 

@unpack k_IN_70SIC, KM_mRNA, KM_50S, k_IN_IF2D, k_on_30S_IF3, k_off_30SIF3, k_on_30SIF3_IF1, k_off_30SIF3IF1, k_on_30SIF3_IF2, k_off_30SIF3IF2, k_on_30SIF3IF1_IF2, k_off_30SIF3IF1IF2, k_on_30SIF3IF2_IF1, k_off_30SIF3IF2IF1, k_on_30SIF123_fMet, k_off_30SIF123fMet = tags.p_met.IN
@unpack k_EFGGDP_Diss, k_EFG_GDP_Ass, k_EFG_GTP_Ass, k_EFGGTP_Diss, k_EFTsf, k_EFTsr, Keq_EFTu, KM_GTP, KM_GDP, KM_EFTuGDP, KM_EFTuGTP, Ki_EFTuGDP, Ki_EFTuGTP, c_EFG_t = tags.p_met.EF
@unpack k_cat, KM_ARS_tRNA, KM_ARS_aa, KM_ARS_ATP = tags.p_met.tRNA
@unpack k_T3j_f, k_T3j_d = tags.p_met.t3 
q_RBS = tags.p_met.q_RBS


# calculation of q_R_elo 
c_RIB = y[tags.idx_first_codon:end]
rna_free = y[tags.idx.RNA]
total_RNA = rna_free + sum(y_rib[1:12])
s70_ini = y[tags.idx.S70]
PTC = y[tags.idx.PTC]

q_R_elo     = ones(gene_sizes);
    
@views for i in 2:(len_sequence - 13)

    if total_RNA-sum(c_RIB[i:i+12]) == 0

        q_R_elo[i] 	= 0;
    else
        q_R_elo[i] 	= 1 - c_RIB[i+12] / (total_RNA-sum(c_RIB[i:i+11]));
    
    end
end 


@views for i in (len_sequence-13) : (len_sequence-1)
    if total_RNA - sum(c_RIB[i : end]) == 0
        q_R_elo[i] = 0
    else 
        q_R_elo[i] = 1 - (c_RIB[end] + PTC) / (total_RNA - sum(c_RIB[i:end-1]))
    end
end


In [23]:

# Calculation of V_TL_EL 
c_Codon = tags.t3_Codon_struc * y[1:tags.Tags] # previously no_of_variables - I hope that tags.Tags is the same


LoadError: MethodError: no method matching (::Colon)(::Int64, ::Vector{SubString{String}})
The function `Colon()` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  (::Colon)(::T, ::Any, [91m::T[39m) where T<:Real
[0m[90m   @[39m [90mBase[39m [90m[4mrange.jl:50[24m[39m
[0m  (::Colon)(::A, ::Any, [91m::C[39m) where {A<:Real, C<:Real}
[0m[90m   @[39m [90mBase[39m [90m[4mrange.jl:10[24m[39m
[0m  (::Colon)(::T, ::Any, [91m::T[39m) where T
[0m[90m   @[39m [90mBase[39m [90m[4mrange.jl:49[24m[39m
[0m  ...


In [ ]:
V_TL_EL = zeros(gene_sizes);
EGT = y[tags.idx.EGT] 
EGT_inv = inv(EGT)
efficiencies = zeros(length(V_TL_EL)-1)
pos_amino = zeros(length(V_TL_EL)-1)
dyn_efac = zeros(length(V_TL_EL)-1)


### Code material

In [ ]:
function four_V_TL_EL(y, tags)
	# Function that is like in analyze_parallel_data but with EGT on y[25]
	# parameters
	len_sequence = length(tags.Seq) # Codon length with stop codon
	idx_first_codon = length(tags.Tag)+1
	gene_sizes = len_sequence-1
	k_cat_Pol, k_D, C_RNP, V_TCmax, K_M_D, C_RNE, k_cat_E, K_M_deg, k_TLEmaxj, KM_T3j, KM_EFGGTP, KM_RK, KM_RF3GTP, k_RF1_diss_GTP, k_RF3_diss_GDP, k_GTP_hydr, k_RF1_ass, k_RF3_GDP_ass, k_RF3_GTP_ass, k_RF3_GDP_diss, k_RF3_GTP_diss, A1, A2, A3, A4, A5, k_cat_K_M_G1, k_cat_K_M_G2, k_RRF, q_RRF, one_q_G2_k_split, k_TE = tags.p_seq
	q_RBS, k_IN_70SIC, KM_mRNA, KM_50S, k_IN_IF2D, k_on_30S_IF3, k_off_30SIF3, k_on_30SIF3_IF1, k_off_30SIF3IF1, k_on_30SIF3_IF2, k_off_30SIF3IF2, k_on_30SIF3IF1_IF2, k_off_30SIF3IF1IF2, k_on_30SIF3IF2_IF1, k_off_30SIF3IF2IF1, k_on_30SIF123_fMet, k_off_30SIF123fMet, k_EFGGDP_Diss, k_EFG_GDP_Ass, k_EFG_GTP_Ass, k_EFGGTP_Diss, k_EFTsf, k_EFTsr, Keq_EFTu, KM_GTP, KM_GDP, KM_EFTuGDP, KM_EFTuGTP, Ki_EFTuGDP, Ki_EFTuGTP, c_EFG_t, k_cat, KM_ARS_tRNA, KM_ARS_aa, KM_ARS_ATP, k_T3j_f, k_T3j_d = tags.p_met;

	# calculation of q_R_elo 
	q_R_elo = ones(Float64, gene_sizes);
	y_rib = y[idx_first_codon:end]
	rna_free = y[tags.idx.RNA]
	total_RNA = rna_free + sum(y_rib[1:12])
	s70_ini = y[tags.idx.S70]
	ptc = y[tags.idx.PTC]

	q_R_elo[1] = 1 - (sum(y_rib[1:12]) +s70_ini)/(rna_free+sum(y_rib[1:12]))

	for i in 2:(gene_sizes-13)
		q_R_elo[i] = 1 - y_rib[i+12] / (total_RNA-sum(y_rib[i:i+11]))
	end

	for i in gene_sizes-12:gene_sizes
		q_R_elo[i] = 1 - (y_rib[i]) / (total_RNA - sum(y_rib[i:gene_sizes])-ptc)
	end

	# Calculation of V_TL_EL 
	c_Codon = tags.t3_Codon_struc * y[1:tags.no_of_variables]
	V_TL_EL = zeros(gene_sizes);
	EGT = y[tags.idx.EGT] 
	EGT_inv = inv(EGT)
	efficiencies = zeros(length(V_TL_EL)-1)
	pos_amino = zeros(length(V_TL_EL)-1)
	dyn_efac = zeros(length(V_TL_EL)-1)

	for i in 1:length(V_TL_EL)-1
    	u = findfirst(x -> x == String(tags.Seq[i+1]), tags.cod)
    	k = findfirst(x -> x == String(tags.Seq[i]), tags.cod)

		efficiency = tags.efac[k] * min(tags.efac_pos_aa[i], tags.efac_int_sd[i])
		efficiencies[i] = efficiency
		pos_amino[i] = tags.efac_pos_aa[i]
		dyn_efac[i] = tags.efac[k]
		V_TL_EL[i+1] = q_R_elo[i] * efficiency * k_TLEmaxj * y_rib[i] /(1+ KM_T3j/c_Codon[u]+ KM_EFGGTP*EGT_inv)
	end

	return V_TL_EL, efficiencies, pos_amino, dyn_efac
end


function V_EL_smooth_analyze_parallel_data(tags, V_TL_EL, y_rib)

	# parameters and arrays 
	len_sequence 	= length(tags.Seq) # Codon length with stop codon
	L 				= len_sequence - 2
	V_ELO 			= zeros(L)
	V_EL_smooth  	= zeros(L)
	h1				= zeros(L)

	# V_TL_EL per second
	V_ELO = (V_TL_EL[2:end]./60)./(y_rib[1:end-1]);
	V_EL_smooth[1:3] 	= [mean(V_ELO[1:i+3])   for i in 1:3]
	V_EL_smooth[4:L-3] 	= [mean(V_ELO[i-3:i+3]) for i in 4:L-3]
	V_EL_smooth[L-2:L] 	= [mean(V_ELO[i-2:L])   for i in L-2:L];

	# Plot
	h1 .= mean(V_ELO)
	
	return V_EL_smooth, h1, V_ELO
end

In [ ]:



df_list = sequence_dataframes(sol, local_tags_list);